# Dash Showcase

We will build a simple interactive dashboard to answer our Citi Bike questions.


**Learning objectives**
- Understand the Dash mental model (layout + callbacks)
- Build a small app with filters and multiple charts
- Run Dash in Colab safely

**Estimated time:** 60-90 minutes


## Setup


In [1]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if (BASE_DIR / "viz-workshop").exists():
    BASE_DIR = BASE_DIR / "viz-workshop"
elif (BASE_DIR / "notebooks").exists():
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR / "../../src"))

from data_prep import load_or_create_dataset, get_paths, print_environment_hint

print_environment_hint()
paths = get_paths(BASE_DIR)

Running locally. Paths are relative to the project folder.


In [2]:
# Load or build the processed dataset (cached after first run)
df = load_or_create_dataset(base_dir=BASE_DIR)
print(df.shape)
df.head()

(200000, 15)


,started_at,ended_at,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,rideable_type,member_casual,duration_min,date,hour,day_of_week,day_name
0,2025-12-13 07:43:49.556,2025-12-13 07:51:15.294,Washington St & Laight St,River Ter & Warren St,40.722310,-74.010600,40.717599,-74.015880,classic_bike,member,7.428967,2025-12-13,7,5,Saturday
1,2025-12-21 13:57:34.955,2025-12-21 14:08:36.049,E 7 St & Ave B,1 Ave & E 44 St,40.725129,-73.981317,40.750020,-73.969053,electric_bike,member,11.018233,2025-12-21,13,6,Sunday
2,2025-12-01 17:35:35.005,2025-12-01 17:59:39.517,Queens Plaza North & Crescent St,34 Ave & 74 St,40.751102,-73.940737,40.752970,-73.892360,classic_bike,member,24.075200,2025-12-01,17,0,Monday
3,2025-12-02 23:01:05.402,2025-12-02 23:04:00.373,Metropolitan Ave & Meeker Ave,N 6 St & Bedford Ave,40.714133,-73.952344,40.717452,-73.958509,electric_bike,member,2.916183,2025-12-02,23,1,Tuesday
4,2025-12-06 16:14:28.161,2025-12-06 16:27:07.386,W 92 St & Broadway,W 76 St & Columbus Ave,40.792100,-73.973900,40.780184,-73.977285,classic_bike,member,12.653750,2025-12-06,16,5,Saturday


## Dash mental model

Dash apps have two parts:
1. **Layout**: what the page looks like.
2. **Callbacks**: how components update when inputs change.


## Build the app layout


In [14]:
import pandas as pd
import plotly.express as px

from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output

# Prepare dropdown options
member_options = ["all"] + sorted(df["member_casual"].dropna().unique().tolist())
ride_options = ["all"] + sorted(df["rideable_type"].dropna().unique().tolist())

app = JupyterDash(__name__)

app.layout = html.Div([
    html.H1("Citi Bike Usage Patterns"),
    html.P("Interactive view of trips by date, rider type, and stations."),

    html.Div([
        html.Label("Membership type"),
        dcc.Dropdown(member_options, value="all", id="member_filter"),
    ]),

    html.Div([
        html.Label("Rideable type"),
        dcc.Dropdown(ride_options, value="all", id="ride_filter"),
    ]),

    html.Div([
        html.Label("Date range"),
        dcc.DatePickerRange(
            id="date_filter",
            min_date_allowed=df["date"].min(),
            max_date_allowed=df["date"].max(),
            start_date=df["date"].min(),
            end_date=df["date"].max(),
        ),
    ], style={"marginBottom": "20px"}),

    dcc.Graph(id="timeseries"),
    dcc.Graph(id="top_stations"),
    dcc.Graph(id="station_map"),
])

/home/narustamyan/test/.venv/lib/python3.12/site-packages/dash/dash.py:642: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



## Add one simple callback


In [15]:
@app.callback(
    Output("timeseries", "figure"),
    Input("member_filter", "value"),
    Input("ride_filter", "value"),
    Input("date_filter", "start_date"),
    Input("date_filter", "end_date"),
)
def update_timeseries(member_value, ride_value, start_date, end_date):
    filtered = df.copy()
    if member_value != "all":
        filtered = filtered[filtered["member_casual"] == member_value]
    if ride_value != "all":
        filtered = filtered[filtered["rideable_type"] == ride_value]
    if start_date:
        filtered = filtered[filtered["date"] >= pd.to_datetime(start_date).date()]
    if end_date:
        filtered = filtered[filtered["date"] <= pd.to_datetime(end_date).date()]

    if filtered.empty:
        return px.line(title="No data for selection")

    trips_by_day = filtered.groupby("date").size().reset_index(name="trips")
    fig = px.line(trips_by_day, x="date", y="trips", title="Daily Trips")
    return fig

## Add more callbacks for stations and map


In [ ]:
@app.callback(
    Output("top_stations", "figure"),
    Output("station_map", "figure"),
    Input("member_filter", "value"),
    Input("ride_filter", "value"),
    Input("date_filter", "start_date"),
    Input("date_filter", "end_date"),
)
def update_station_charts(member_value, ride_value, start_date, end_date):
    filtered = df.copy()
    if member_value != "all":
        filtered = filtered[filtered["member_casual"] == member_value]
    if ride_value != "all":
        filtered = filtered[filtered["rideable_type"] == ride_value]
    if start_date:
        filtered = filtered[filtered["date"] >= pd.to_datetime(start_date).date()]
    if end_date:
        filtered = filtered[filtered["date"] <= pd.to_datetime(end_date).date()]

    if filtered.empty:
        empty_fig = px.bar(title="No data for selection")
        return empty_fig, px.scatter_mapbox(title="No data")

    station_counts = (
        filtered.dropna(subset=["start_station_name"])
                .groupby("start_station_name")
                .size()
                .sort_values(ascending=False)
                .head(50)
                .reset_index(name="trips")
    )
    bar_fig = px.bar(station_counts, x="trips", y="start_station_name", orientation="h", title="Top Start Stations")

    stations = (
        filtered.dropna(subset=["start_lat", "start_lng", "start_station_name"])
                .groupby(["start_station_name", "start_lat", "start_lng"])
                .size()
                .reset_index(name="trips")
                .sort_values("trips", ascending=False)
                .head(200)
    )

    map_fig = px.scatter_map(
        stations,
        lat="start_lat",
        lon="start_lng",
        size="trips",
        hover_name="start_station_name",
        zoom=10,
        title="Top Stations (map)"
    )
    return bar_fig, map_fig

## Run the app

In Colab, JupyterDash is often the simplest option. If inline rendering fails, use the provided link in the output.


In [17]:
# In Colab, mode="inline" usually works. If you get a blank area,
# rerun and open the provided link.
app.run()

## Troubleshooting Dash in Colab
- If you see a blank output, rerun the cell and open the provided link.
- If the port is busy, change `port=8050` to another number.
- If you get dependency errors, re-run the install cell.


## Exercises
1. Add an hour range slider and filter the dataset by hour.
2. Add a download button to export the filtered data. (Optional stretch)

**Check yourself:** the new control should update at least one chart.


**Solution (optional): hour range filter (snippet)**

In [ ]:
# Add this to the layout:
# dcc.RangeSlider(0, 23, 1, value=[7, 19], id="hour_filter")

# Then add hour_filter as Input to callbacks and filter like this:
# if hour_range:
#     filtered = filtered[(filtered["hour"] >= hour_range[0]) & (filtered["hour"] <= hour_range[1])]

## Common pitfalls
- Empty selections: return a friendly empty figure instead of crashing.
- Colab rendering: if inline mode is blank, open the provided external link.


## Wrap-up
You built a working dashboard that reuses the same dataset and tells a consistent story. Great job.

**Next steps:** consider adding more insights or deploying the app.
